# LuantiVoxel — automated GPU visual validation

Opens the real game, renders it from fixed viewpoints at **1920x1080**, captures the
actual framebuffer, and packages everything into a ZIP you can download.

No playing, no clicking, no remote desktop. Press **Runtime > Run all**.

**It refuses to lie.** If the GPU cannot be positively identified as hardware —
llvmpipe, SwiftShader, swrast, anything software — the run stops with
`SOFTWARE_RENDERING_DETECTED` and produces no screenshots. A software render of a
voxel landscape looks perfectly plausible and is worthless as evidence of what the
game looks like on a GPU, so it is treated as a failure rather than a result.

In [ ]:
# ---------------------------------------------------------------------------
# CONFIG — change REPO_URL if you have a fork
# ---------------------------------------------------------------------------
REPO_URL = "https://github.com/Daniel-debug-boop/luanti.git"
REF      = "master"
GODOT_VERSION = "4.4-stable"

RESOLUTION  = "1920x1080"
FRAMES      = 120
WARMUP      = 30
OUTPUT_DIR  = "render-test-results"

# Both presets are run. "baseline" strips POM, the stochastic mapping,
# SSIL and volumetric fog so the captures show GEOMETRY -- that is the
# one you diagnose a broken mesh in, because post-processing invents
# detail over missing faces and hides the fault. "game" is the shipping
# look. A fault visible in both is a mesh bug; one visible only in the
# game preset is an effect bug.
EFFECTS = ["baseline", "game", "slope"]

# GPU detection is a hard gate. Leave this True.
REQUIRE_HARDWARE_GPU = True

import os, sys, subprocess, json, textwrap

def sh(cmd, check=True, **kw):
    print(f"$ {cmd}", flush=True)
    p = subprocess.run(cmd, shell=True, text=True,
                       capture_output=True, **kw)
    if p.stdout: print(p.stdout)
    if p.stderr: print(p.stderr, file=sys.stderr)
    if check and p.returncode != 0:
        raise SystemExit(f"command failed ({p.returncode}): {cmd}")
    return p

print("config ok")

## 1 — Environment

Prints what this machine actually is. It does **not** assume NVIDIA: Colab may hand you
a T4, an L4, an A100, or a CPU-only runtime, and the pipeline has to work with whichever
device it is given or refuse.

In [ ]:
print("=" * 70)
print("OS        :", sh("uname -a", check=False).stdout.strip())
print("python    :", sys.version.split()[0])

# Every probe is optional and guarded. Absence of one is information, not an error.
for tool, args in [("nvidia-smi", "nvidia-smi"),
                  ("lspci", "lspci | grep -iE 'vga|3d|display' || true"),
                  ("dri", "ls -l /dev/dri 2>&1 || true"),
                  ("glxinfo", "which glxinfo || echo 'not installed'")]:
    print(f"\n--- {tool} ---")
    print(sh(args, check=False).stdout.strip() or "(nothing)")

## 2 — Clone the repository

In [ ]:
REPO_DIR = "/content/luantivoxel"
if not os.path.isdir(REPO_DIR):
    sh(f"git clone --depth 1 --branch {REF} {REPO_URL} {REPO_DIR}")
else:
    sh(f"git -C {REPO_DIR} pull --ff-only origin {REF}")

GIT_COMMIT = sh(f"git -C {REPO_DIR} rev-parse HEAD").stdout.strip()
print("commit:", GIT_COMMIT)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

## 3 — Dependencies

The game is the **Godot 4.4 client** in `godot_client/`, so the C++ engine in `src/` is
not needed and is not compiled — that would be minutes of build time for a component the
benchmark does not run.

What is needed is the Godot binary, its **export templates** (to produce a standalone
build rather than a project folder), and Xvfb plus Mesa for a display to render into.
All of it comes from the project's own tooling — see `godot_client/tools/build_release.sh`.

In [ ]:
sh("apt-get update -qq && apt-get install -y -qq "
   "xvfb libgl1-mesa-dri libglx-mesa0 libxcursor1 libxinerama1 "
   "libxrandr2 libxi6 libxext6 libx11-6 unzip wget", check=False)
# Cell 7 verifies capture dimensions with Pillow. It is preinstalled on
# Colab, but this notebook also has to run on a runtime that cleared it,
# and a missing module there would look like a rendering failure.
sh("pip install -q Pillow", check=False)

GODOT_DIR = "/opt/godot"
os.makedirs(GODOT_DIR, exist_ok=True)
GODOT = f"{GODOT_DIR}/godot"

if not os.path.exists(GODOT):
    url = (f"https://github.com/godotengine/godot/releases/download/"
           f"{GODOT_VERSION}/Godot_v{GODOT_VERSION}_linux.x86_64.zip")
    sh(f"wget -q {url} -O /tmp/godot.zip && unzip -o -q /tmp/godot.zip -d {GODOT_DIR}")
    sh(f"mv {GODOT_DIR}/Godot_v{GODOT_VERSION}_linux.x86_64 {GODOT} && chmod +x {GODOT}")

print(sh(f"{GODOT} --version").stdout.strip())

## 3b — GPU drivers and the Vulkan loader (T4)

A Colab T4 runtime has the NVIDIA kernel driver loaded, but that is not the
same as having a **Vulkan device the game can open**. Godot 4 is a Vulkan
renderer: without the loader and an ICD that points at the NVIDIA library,
every driver in turn fails and the GPU gate below refuses the run.

So the loader, the Mesa fallback and a diagnostic tool are installed, the
NVIDIA ICD is written if the driver libraries are present, and `vulkaninfo`
is asked what it can actually see. The cell prints the answer rather than
assuming it.


In [ ]:
import glob, os, json
# ---------------------------------------------------------------------------
# GPU drivers: Vulkan loader + an ICD the game can open
# ---------------------------------------------------------------------------
# The NVIDIA kernel module being loaded does NOT mean a Vulkan device exists.
# Godot is a Vulkan renderer, so without libvulkan + an ICD pointing at the
# NVIDIA library every --rendering-driver in turn fails and the gate refuses.
sh("apt-get install -y -qq libvulkan1 mesa-vulkan-drivers vulkan-tools "
   "libgl1-mesa-dri libglx-mesa0 libegl-mesa0 libgbm1", check=False)

# Register the NVIDIA ICD when the driver libraries are on the image but the
# manifest is not. Colab ships the .so files; the JSON that tells the loader
# they exist is what is usually missing.
ICD_DIR = "/usr/share/vulkan/icd.d"
os.makedirs(ICD_DIR, exist_ok=True)
ICD = os.path.join(ICD_DIR, "nvidia_icd.json")
libs = glob.glob("/usr/lib/x86_64-linux-gnu/libGLX_nvidia.so.*") \
     + glob.glob("/usr/lib/x86_64-linux-gnu/libnvidia-glcore.so.*")
if libs and not os.path.exists(ICD):
    with open(ICD, "w") as fh:
        json.dump({"file_format_version": "1.0.0",
                   "ICD": {"library_path": "libGLX_nvidia.so.0",
                           "api_version": "1.3.194"}}, fh, indent=2)
    print("wrote", ICD, "->", libs[0])
elif os.path.exists(ICD):
    print("nvidia ICD already present:", ICD)
else:
    print("no NVIDIA driver libraries found; Vulkan will fall back to Mesa")

print("\n--- vulkaninfo (summary) ---")
vi = sh("vulkaninfo --summary 2>&1 | head -60", check=False)
print(vi.stdout or "")

# The gate that matters: is there any Vulkan device at all? Presence of the
# loader without a device is the failure this cell exists to surface early,
# with a message that says which of the two is missing.
probe = sh("vulkaninfo --summary 2>&1", check=False).stdout or ""
devices = [ln for ln in probe.splitlines()
           if "deviceName" in ln or "GPU id" in ln]
if devices:
    print("Vulkan devices:")
    for ln in devices:
        print("  ", ln.strip())
else:
    print("WARNING: vulkaninfo reports no Vulkan device. The GPU gate in the "
          "next cells will refuse the run; that is the gate working, not a "
          "notebook error.")

print("driver setup done")


## 4 — Build

Uses the project's own `build_release.sh`, which exports the Linux build **and
smoke-tests the resulting binary**, failing if it does not reach a ready state or logs a
script error. A build that starts and then falls over never reaches the render test.

In [ ]:
TPL_DIR = os.path.expanduser(
    "~/.local/share/godot/export_templates/4.4.stable")
os.makedirs(TPL_DIR, exist_ok=True)

if not os.path.exists(f"{TPL_DIR}/linux_release.x86_64"):
    tpl = (f"https://github.com/godotengine/godot/releases/download/"
           f"{GODOT_VERSION}/Godot_v{GODOT_VERSION}_export_templates.tpz")
    sh(f"wget -q {tpl} -O /tmp/tpl.tpz && unzip -o -q /tmp/tpl.tpz -d /tmp/tpl")
    for f in ("linux_release.x86_64", "linux_debug.x86_64", "version.txt"):
        sh(f"cp /tmp/tpl/templates/{f} {TPL_DIR}/")

sh(f"sh godot_client/tools/build_release.sh {GODOT}")

BUILD_DIR = os.path.join(REPO_DIR, "build", "luantivoxel")
EXE = os.path.join(BUILD_DIR, "luantivoxel.x86_64")
PCK = os.path.join(BUILD_DIR, "luantivoxel.pck")
print("exe:", EXE, os.path.exists(EXE))
print("pck:", PCK, os.path.exists(PCK))

## 5 — GPU verification (hard gate)

Starts a virtual display and asks the **actual game binary** what adapter it is on.
This is the same code path the render test uses, not a separate check that could
disagree with it.

In [ ]:
os.environ["DISPLAY"] = ":99"
os.environ.setdefault("HOME", "/root")
sh("pkill Xvfb 2>/dev/null; Xvfb :99 -screen 0 1920x1080x24 >/tmp/xvfb.log 2>&1 &")
import time; time.sleep(3)

def gpu_gate():
    """Ask the game itself, trying each driver in turn."""
    best = ("", "")
    for drv in DRIVERS:
        print(f"\n########## trying rendering driver: {drv} ##########")
        out = os.path.join(os.getcwd(), "_gpuprobe")
        sh(f"rm -rf {out}")
        try:
            p = subprocess.run(
                [EXE, "--rendering-driver", drv, "--resolution", RESOLUTION,
                 "--", "--render-test", "--frames", "1",
                 "--warmup-frames", "0", "--output", out],
                text=True, capture_output=True, timeout=900,
                env={**os.environ})
        except subprocess.TimeoutExpired:
            print(f"{drv}: timed out")
            continue
        txt = p.stdout + p.stderr
        for line in txt.splitlines():
            if "render-test" in line:
                print(line)
        report = os.path.join(out, "gpu-info.txt")
        body = open(report).read() if os.path.exists(report) else ""
        cls = ""
        for line in body.splitlines():
            if line.startswith("classification"):
                cls = line.split(None, 1)[1].strip()
        if body:
            print(body)
        if cls == "hardware":
            print(f"\n{drv} gave real hardware acceleration. Using it.\n")
            return drv, cls
        best = (drv, cls)
    return best

DRIVERS = ["vulkan", "opengl3"]
chosen, cls = gpu_gate()
os.environ["RENDER_DRIVER"] = chosen or "vulkan"
print(f"\ndriver chosen = {os.environ['RENDER_DRIVER']!r}, "
      f"classification = {cls!r}")

if REQUIRE_HARDWARE_GPU and cls != "hardware":
    raise SystemExit(
        "Hardware GPU rendering was not detected. Refusing to produce a "
        f"misleading final visual-validation result (classification={cls!r}).\n"
        "\nFix: Runtime > Change runtime type > GPU, then restart. 'GPU' in the "
        "Colab picker does not always mean a device is attached, and Colab GPU "
        "availability varies. Cell 1 shows what the runtime actually has.")

## 6 — Render test

1920x1080, four fixed viewpoints, real framebuffer, real assets, real shaders.

In [ ]:
sh(f"rm -rf {OUTPUT_DIR}")
os.makedirs(os.path.join(OUTPUT_DIR, "captures"), exist_ok=True)

# Two runs, one per effect preset, into the same output tree. Each is
# tagged by filename so the two are never confused for one another.
for preset in EFFECTS:
    tag = {"baseline": "", "game": "_game", "slope": "_slope"}[preset]
    cmd = [EXE, "--rendering-driver", os.environ.get("RENDER_DRIVER", "vulkan"),
           "--resolution", RESOLUTION, "--", "--render-test",
           "--scene", "benchmark", "--resolution", RESOLUTION,
           "--frames", str(FRAMES), "--warmup-frames", str(WARMUP),
           "--all-cameras", "--no-ui", "--effects", preset,
           "--output", os.path.join(OUTPUT_DIR, "effects-" + preset)]
    print("$ " + " ".join(cmd))
    proc = subprocess.run(cmd, text=True, capture_output=True, timeout=3600,
                          env={**os.environ})
    print(proc.stdout[-4000:])
    if proc.returncode != 0:
        print(proc.stderr[-3000:])
        raise SystemExit(
            f"render test ({preset}) failed with exit code {proc.returncode}")

# Lift the per-preset captures into one flat captures/ directory, suffixing
# the game-preset files so "front.png" is always the diagnostic one.
import glob, shutil
for preset in EFFECTS:
    tag = {"baseline": "", "game": "_game", "slope": "_slope"}[preset]
    for p in sorted(glob.glob(os.path.join(
            OUTPUT_DIR, "effects-" + preset, "captures", "*.png"))):
        shutil.copy(p, os.path.join(OUTPUT_DIR, "captures",
                                    os.path.basename(p)[:-4] + tag + ".png"))
# The baseline run owns the top-level artefacts; the game run's summary is
# kept alongside it so neither is lost.
# The baseline run is the primary one: it is the diagnostic capture set,
# so its metadata and content report are the ones cells 7 and 9 read. The
# game run's copies are kept alongside, suffixed, so neither is lost.
for extra in ("metadata.json", "performance.json", "performance.txt",
              "gpu-info.txt", "render-log.txt", "capture-content.txt"):
    src_p = os.path.join(OUTPUT_DIR, "effects-baseline", extra)
    if os.path.exists(src_p):
        shutil.copy(src_p, os.path.join(OUTPUT_DIR, extra))
    src_p = os.path.join(OUTPUT_DIR, "effects-game", extra)
    if os.path.exists(src_p):
        shutil.copy(src_p, os.path.join(OUTPUT_DIR, extra + ".game"))
    src_p = os.path.join(OUTPUT_DIR, "effects-slope", extra)
    if os.path.exists(src_p):
        shutil.copy(src_p, os.path.join(OUTPUT_DIR, extra + ".slope"))

for required in ("metadata.json", "capture-content.txt"):
    if not os.path.exists(os.path.join(OUTPUT_DIR, required)):
        raise SystemExit(f"run did not produce {required}")
print("render test exited 0 for:", ", ".join(EFFECTS))


## 7 — Validate

Refuses to package a result that is incomplete or the wrong size. A screenshot that is
not genuinely 1920x1080 is worse than no screenshot, because it looks like one.

In [ ]:
from PIL import Image
import math

W, H = (int(x) for x in RESOLUTION.split("x"))
cap_dir = os.path.join(OUTPUT_DIR, "captures")
shots = sorted(f for f in os.listdir(cap_dir) if f.endswith(".png")) \
    if os.path.isdir(cap_dir) else []
problems = []

if not shots:
    problems.append("SCREENSHOT_CAPTURE_FAILURE: no captures produced")


def judge(path):
    """Independent second opinion on the pixels.

    The game already measures its own frames and writes capture-content.txt.
    This recomputes the same quantities here, in Python, from the PNGs that
    actually made it into the ZIP -- so a bug in the C#-side measurement
    cannot quietly pass the whole pipeline.
    """
    im = Image.open(path).convert("RGB")
    px = list(im.getdata())
    n = len(px)
    # PIL gives 0..255 channel values; the thresholds below are
    # written against 0..1 luma, so normalise here. Getting this
    # wrong makes the washed-out test unable to fire at all,
    # which is how a broken frame passes validation.
    luma = [(0.299 * r + 0.587 * g + 0.114 * b) / 255.0
            for r, g, b in px]
    mean = sum(luma) / n
    sd = math.sqrt(sum((v - mean) ** 2 for v in luma) / n)
    cols = len({(r >> 3, g >> 3, b >> 3) for r, g, b in px})
    return mean, sd, cols


for s in shots:
    p = os.path.join(cap_dir, s)
    w, h = Image.open(p).size
    ok = (w, h) == (W, H)
    print(f"  {'ok  ' if ok else 'BAD '} {s:<22} {w}x{h}")
    if not ok:
        problems.append(f"INVALID_RESOLUTION: {s} is {w}x{h}, expected {W}x{H}")
        continue
    mean, sd, cols = judge(p)
    print(f"       colours={cols:<8} luma={mean:.3f} +/- {sd:.3f}")
    # Same thresholds as RenderTest.judge_capture, in 0..1 luma units.
    verdict = ""
    if cols < 8:
        verdict = f"only {cols} distinct colour(s): flat fill, nothing rendered"
    elif sd < 0.01:
        verdict = f"luma stddev {sd:.4f} < 0.01: no visible structure"
    elif mean > 0.82 and sd < 0.15:
        verdict = (f"mean luma {mean:.3f} with stddev {sd:.4f}: washed out, "
                   "fog/bloom over broken geometry")
    if verdict:
        print(f"       REJECTED: {verdict}")
        problems.append(f"{s}: {verdict}")

for required in ("metadata.json", "performance.json", "performance.txt",
                 "gpu-info.txt", "render-log.txt", "capture-content.txt"):
    p = os.path.join(OUTPUT_DIR, required)
    if not os.path.exists(p):
        problems.append(f"missing artefact: {required}")
    else:
        print(f"  ok   {required}")

meta = json.load(open(os.path.join(OUTPUT_DIR, "metadata.json")))
if not meta.get("hardware_acceleration"):
    problems.append("HARDWARE_GPU: FAIL - metadata does not claim acceleration")


# --- negative control ------------------------------------------------------
# A validator that has only ever seen good frames is indistinguishable from
# one that always passes. Feed it the shape of the frame a real GPU run
# actually produced -- a bright, near-uniform haze over geometry whose faces
# were all culled -- and require it to be rejected. If this ever passes
# silently, the checks above are decorative.
haze = Image.new("RGB", (W, H))
for y in range(0, H, 4):
    for x in range(0, W, 4):
        v = 0.90 + 0.01 * math.sin(x * 0.01) + 0.01 * math.cos(y * 0.013)
        haze.paste((int(v * 250), int(v * 252), int(v * 255)),
                   (x, y, x + 4, y + 4))
haze_path = os.path.join(OUTPUT_DIR, "_self_test_broken_frame.png")
haze.save(haze_path)
h_mean, h_sd, h_cols = judge(haze_path)
ctl_rejected = (h_cols < 8 or h_sd < 0.01 or (h_mean > 0.82 and h_sd < 0.15))
print(f"\nself-test: a known-broken frame gives colours={h_cols} "
      f"luma={h_mean:.3f}+/-{h_sd:.4f} -> "
      f"{'REJECTED (correct)' if ctl_rejected else 'ACCEPTED (validator is weak)'}")
if not ctl_rejected:
    problems.append("validator self-test: a broken frame was accepted")
os.remove(haze_path)

if problems:
    raise SystemExit("VALIDATION FAILED:\n  " + "\n  ".join(problems))
print("\nvalidation passed")


## 7b — Slope-blend validation (hard gate)

The only check that can happen here rather than on a CPU. A
shading-language error in `voxel_slope_blend.gdshader` is invisible to
every headless check, because headless Godot never compiles shaders.
This cell refuses to report a benchmark if the shader did not compile or
if the slope mapping was never selected.


In [ ]:
# ---------------------------------------------------------------------------
# Slope-blend validation (Phase 2.2)
# ---------------------------------------------------------------------------
# The one check that can only happen here. A shading-language error in
# voxel_slope_blend.gdshader is invisible to every CPU-side check: headless
# Godot uses the dummy renderer, so `Shader.set_code` never reaches a driver
# and the file is never compiled. On a real GPU it either compiles or the
# surface renders wrong, and this is where that is caught.
#
# It is deliberately a hard failure. A benchmark that quietly photographs a
# magenta terrain and reports FPS for it is worse than no benchmark.

SLOPE_LOG = os.path.join(OUTPUT_DIR, "render-log.txt.slope")
if not os.path.exists(SLOPE_LOG):
    raise SystemExit(
        "the slope preset produced no log, so the slope shader was never "
        "loaded -- check that EFFECTS includes 'slope'")

log = open(SLOPE_LOG, errors="replace").read()

# The preset has to have actually selected the mapping, or a clean log means
# nothing: an unrun shader also produces no errors.
want = "mapping=slope"
if want not in log:
    raise SystemExit(
        f"the slope run never selected the slope mapping ({want!r} absent "
        f"from the log), so a clean log would prove nothing")

# Godot's shader compile failures, matched loosely on purpose: the exact
# wording differs between drivers and between Vulkan and OpenGL.
signatures = ["shader compilation failed", "shader error", "shader compile",
              "error in shader", "unable to compile shader"]
lowered = log.lower()
hits = [s for s in signatures if s in lowered]
if hits:
    bad = [ln for ln in log.splitlines()
           if any(s in ln.lower() for s in signatures)]
    print("\n".join(bad[:40]))
    raise SystemExit(
        "the slope shader did not compile on this GPU "
        f"({', '.join(hits)}). The captures from the slope preset are not "
        "valid evidence of anything.")

print("slope shader: selected the slope mapping and reported no compile "
      "errors on this driver")
print("slope captures:", sorted(f for f in os.listdir(cap_dir)
                               if f.endswith("_slope.png")))


## 8 — Package

In [ ]:
import datetime, zipfile

sh(f"echo {GIT_COMMIT} > {OUTPUT_DIR}/git-commit.txt")
sh(f"({GODOT} --version; echo; nvidia-smi 2>/dev/null || true) "
   f"> {OUTPUT_DIR}/build-info.txt")

README = '''LuantiVoxel GPU render test
============================
Automated visual validation of the Godot 4.4 client.

These are real frames from the shipping renderer at 1920x1080 on a hardware
GPU. An llvmpipe/software result is NOT a valid visual validation and this
pipeline refuses to produce one.

The seed is recorded in metadata.json; the same seed reproduces the same
terrain on any machine, so two runs are directly comparable.
'''
open(os.path.join(OUTPUT_DIR, "README.txt"), "w").write(README)

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
ZIP = f"LUANTIVOXEL_GPU_RENDER_TEST_{stamp}.zip"

# Only validation artefacts. No build dirs, no .git, no caches.
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _dirs, files in os.walk(OUTPUT_DIR):
        for f in files:
            full = os.path.join(root, f)
            z.write(full, os.path.relpath(full, os.path.dirname(OUTPUT_DIR)))

print("zip:", ZIP, os.path.getsize(ZIP), "bytes")
for n in sorted(zipfile.ZipFile(ZIP).namelist()):
    print("  ", n)

## 9 — Results

In [ ]:
perf = json.load(open(os.path.join(OUTPUT_DIR, "performance.json")))
fps = perf.get("average_fps")

print("=" * 56)
print("LUANTIVOXEL GPU RENDER TEST")
print("=" * 56)
print(f"GPU                : {meta.get('gpu_vendor')} / {meta.get('gpu_name')}")
print(f"Graphics API       : {meta.get('graphics_api')}")
print(f"Rendering device   : {meta.get('rendering_device')}")
print(f"Hardware accel     : {'PASS' if meta.get('hardware_acceleration') else 'FAIL'}")
print(f"Capture content    : {'PASS' if meta.get('capture_content_pass') else 'FAIL'}")
print(f"Resolution         : {meta.get('resolution')}")
print(f"Scene / seed       : {meta.get('scene')} / {meta.get('seed')}")
print(f"Effects preset     : {meta.get('effects')} (files ending _game are the shipping-look run)")
print(f"Frames             : {perf.get('frames_measured')} measured, "
      f"{perf.get('warmup_frames')} warm-up")
print(f"Average frame time : "
      f"{perf['average_frame_ms']:.2f} ms" if perf.get('average_frame_ms')
      else "Average frame time : unavailable")
print(f"Average FPS        : {fps:.2f}" if fps else "Average FPS        : unavailable")
print(f"GPU frame time     : {perf.get('gpu_frame_ms') or 'unavailable'}")
print(f"Screenshots        : {', '.join(shots)}")
print(f"Commit             : {GIT_COMMIT[:12]}")
print("=" * 56)
from IPython.display import display, Image as IPImage, HTML
display(HTML(f'<a href="{ZIP}" download>Download {ZIP}</a>'))
for s in shots:
    display(IPImage(filename=os.path.join(cap_dir, s)))